# MQM-Core Error Typology — interactive explorer

This notebook walks **step by step** through how the self-contained, interactive HTML view of the
[MQM-Core](https://themqm.org/) translation-quality error typology is generated. The heavy lifting
lives in the reusable module [`src/mqm_viz`](../src/mqm_viz/__init__.py) and in
[`template.html`](../template.html); here we use it with narration. That same module is what
[`build.py`](../build.py) runs in the GitHub Pages workflow, so **there is no duplicated logic**.

The flow has three steps:

1. download the official **MQM-Full** spreadsheet from <https://themqm.org/downloads/>,
2. parse the `MQMFull Master` sheet (standard library only — no `openpyxl`/`pandas`), and
3. render a collapsible interactive tree into a single self-contained HTML file.

**What the HTML lets you do**

- Browse the typology as a collapsible tree; **node color = error level**
  (dimension → N1 → N2 → N3) and **filled vs. outline = Core vs. Extension**.
- Hover a node to see its description, examples, identifier (PID) and Core/Extension label.
- **Tick the checkbox** on any node to add it to a *selection* (e.g. the error types you want to
  include in an annotation guideline). The selection is grouped by dimension, persists across
  reloads (`localStorage`) and can be exported to **CSV / JSON**.
- Filter the tree with **All / Core only / Selected**.

## Attribution and license

The typology used here is the **MQM Error Typology** (MQM-Core), &copy; **The MQM Council**,
published at <https://themqm.org> and obtained from the MQM-Full spreadsheet at
<https://themqm.org/downloads/>. MQM materials are licensed under the
**Creative Commons Attribution 4.0 International License (CC BY 4.0)** —
<https://creativecommons.org/licenses/by/4.0/>.

**Changes from the source** (as required by CC BY 4.0): the typology is reshaped into a JSON tree
for visualization; two typo'd *parent* references are corrected (`locale-convention` →
`locale-conventions`, `locale-specific-punctuation` → `locale-specific punctuation`); and a
`Core` / `Extension` flag is derived from the PID prefix (`MQMC` = Core, `MQMN` = Extension).
This attribution is also embedded in the generated HTML. This explorer is an independent
adaptation and is **not endorsed by, nor affiliated with, The MQM Council**.

## 0 · Import the module

All the logic lives in `src/mqm_viz`. We add `src/` to the path so it can be imported when the
notebook runs from `notebooks/`.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import mqm_viz

BUILD_DIR = ROOT / "build"
BUILD_DIR.mkdir(exist_ok=True)
print("Repository root:", ROOT)

## 1 · Download the MQM-Full spreadsheet

`download_spreadsheet` fetches the official `.xlsx` (with a browser User-Agent — the server
returns HTTP 406 for a bare `urllib` request) and caches it under `build/`, which git ignores.

In [ ]:
xlsx_path = mqm_viz.download_spreadsheet(BUILD_DIR / "MQMFull_Master-Official.xlsx")
print(f"{xlsx_path} ({xlsx_path.stat().st_size:,} bytes)")

## 2 · Parse the spreadsheet into the typology

An `.xlsx` is just a zip of XML. `build_typology` reads it with `zipfile` + `xml.etree`
(no third-party dependencies), applies the source fixes and derives the `core` flag from the
PID prefix. It also verifies that every *parent* resolves.

Columns in `MQMFull Master`:

| Col. | Spreadsheet header | Content |
|---|---|---|
| A | `Error Type Display Name` | Visible error-type name |
| B | `Error Type Description` | Description |
| C | `Error Type Examples` | Examples |
| D | `Error Type Notes` | Notes |
| E | `Error Type Level #` | Level (depth in the tree) |
| F | `Alphanumeric Error Type PID` | Alphanumeric identifier (PID) |
| G | `Mnemonic Error Type ID` | Mnemonic identifier |
| H | `Error Type Parent` | Parent (mnemonic id) |
| I | `See Note #` | Note reference |

In [ ]:
nodes = mqm_viz.build_typology(xlsx_path)

n_core = sum(n["core"] for n in nodes)
print(f"{len(nodes)} nodes | {n_core} core | {len(nodes) - n_core} extensions")
print("roots:", [n["name"] for n in nodes if not n["parent"]])

# Intermediate JSON, handy for inspection or feeding an exported selection back in.
(BUILD_DIR / "mqm_typology.json").write_text(
    json.dumps(nodes, ensure_ascii=False, indent=2), encoding="utf-8"
)
nodes[0]

## 3 · Render the interactive HTML

`render_html` injects the nodes (as JSON) into the `__DATA__` placeholder of
[`template.html`](../template.html). The result is a single self-contained file (its only
external dependency is D3, loaded from a CDN).

In [ ]:
html = mqm_viz.render_html(nodes)

out_html = BUILD_DIR / "mqm_error_typology.html"
out_html.write_text(html, encoding="utf-8")
print(f"{out_html} ({len(html):,} bytes)")
print("\nOpen it in a browser:")
print("   ", out_html.resolve().as_uri())

## 4 · Preview it here

Optional: embed the view directly in the notebook.

In [ ]:
from IPython.display import IFrame
IFrame(src=out_html.resolve().as_uri(), width="100%", height=600)

## 5 · How to use the explorer

- **Browse**: click a node to expand/collapse; hover to see description, examples, PID and
  Core/Extension.
- **Select**: tick the checkbox next to an error type to add it to *Selected for guideline*.
  The selection is independent (ticking a parent does **not** tick its children), is grouped
  by dimension and is saved in the browser (`localStorage`).
- **Filter**: `Show: All / Core only / Selected`.
- **Export**: copy or download the selection as **CSV** or **JSON**.

To generate the version published to GitHub Pages (at `dist/index.html`), run from the repo
root:

```bash
uv run build.py
```